# Advanced Clustering: HDBSCAN + UMAP Pipeline

We explore **density-based** clustering and **nonlinear dimensionality reduction** on synthetic high-dimensional data, compare reducers (PCA, t-SNE, UMAP) and clusterers (KMeans, DBSCAN, HDBSCAN), then chain **UMAP → HDBSCAN** when those libraries are available.

## Introduction

**KMeans** assumes convex, similarly sized globular clusters. **DBSCAN** finds arbitrary shapes but is sensitive to density variation and fixed `eps`. **HDBSCAN** builds a hierarchy of density estimates and extracts stable clusters—often better for messy real data. **UMAP** tends to preserve both local and global structure better than t-SNE for many workflows, making it a common front-end to density clustering.

In [ ]:
try:
    import numpy as np
    from sklearn.datasets import make_blobs
    from sklearn.preprocessing import StandardScaler
    SKLEARN_OK = True
except ImportError as e:
    SKLEARN_OK = False
    print(e)

try:
    import umap
    UMAP_OK = True
except ImportError:
    UMAP_OK = False
    print("umap-learn not installed; UMAP sections will be skipped.")

try:
    import hdbscan
    HDBSCAN_OK = True
except ImportError:
    HDBSCAN_OK = False
    print("hdbscan not installed; HDBSCAN sections will use notes only.")

SKLEARN_OK, UMAP_OK, HDBSCAN_OK

## Generate synthetic high-dimensional blobs

We use `make_blobs` with many features so linear and nonlinear reduction behave differently.

In [ ]:
if SKLEARN_OK:
    rng = np.random.default_rng(42)
    X, y_true = make_blobs(
        n_samples=800,
        n_features=50,
        centers=5,
        cluster_std=1.2,
        random_state=42,
    )
    # Mild irrelevant dimensions (synthetic noise features)
    noise = rng.normal(0, 0.5, size=X.shape)
    X = X + noise
    X = StandardScaler().fit_transform(X)
    print(X.shape, "unique labels", np.unique(y_true))

## PCA vs t-SNE vs UMAP (2D embedding)

**PCA** is fast and linear. **t-SNE** emphasizes local neighborhoods; **UMAP** often yields more interpretable global separation for downstream clustering. All are fit on the same scaled synthetic data.

In [ ]:
if SKLEARN_OK:
    from sklearn.decomposition import PCA
    from sklearn.manifold import TSNE

    X_pca = PCA(n_components=2, random_state=42).fit_transform(X)
    X_tsne = TSNE(n_components=2, random_state=42, max_iter=500).fit_transform(X)
    if UMAP_OK:
        reducer = umap.UMAP(n_components=2, random_state=42)
        X_umap = reducer.fit_transform(X)
    else:
        X_umap = None

    try:
        import matplotlib.pyplot as plt
    except ImportError:
        print("matplotlib not available; skip plots.")
        plt = None

    if plt is None:
        pass
    else:
        fig, axes = plt.subplots(1, 3 if UMAP_OK else 2, figsize=(12, 4))
        axes[0].scatter(X_pca[:, 0], X_pca[:, 1], c=y_true, s=8, alpha=0.7)
        axes[0].set_title("PCA")
        axes[1].scatter(X_tsne[:, 0], X_tsne[:, 1], c=y_true, s=8, alpha=0.7)
        axes[1].set_title("t-SNE")
        if UMAP_OK:
            axes[2].scatter(X_umap[:, 0], X_umap[:, 1], c=y_true, s=8, alpha=0.7)
            axes[2].set_title("UMAP")
        plt.tight_layout()
        plt.show()

## HDBSCAN vs KMeans vs DBSCAN

We compare cluster assignments in the **original scaled space** (50D). Hyperparameters are chosen for a reasonable baseline on this synthetic set; tune per dataset.

In [ ]:
if SKLEARN_OK:
    from sklearn.cluster import KMeans, DBSCAN
    from sklearn.metrics import adjusted_rand_score, silhouette_score

    km = KMeans(n_clusters=5, random_state=42, n_init="auto").fit_predict(X)
    db = DBSCAN(eps=1.2, min_samples=8).fit_predict(X)
    if HDBSCAN_OK:
        hlabels = hdbscan.HDBSCAN(min_cluster_size=30, min_samples=8).fit_predict(X)
    else:
        hlabels = None

    def report(name, labels):
        mask = labels >= 0
        if mask.sum() < 2 or len(set(labels[mask])) < 2:
            print(name, "ARI", "n/a", "silhouette", "n/a")
            return
        sil = silhouette_score(X[mask], labels[mask])
        ari = adjusted_rand_score(y_true[mask], labels[mask])
        print(f"{name:8} ARI={ari:.3f} silhouette={sil:.3f} n_clusters~{len(set(labels[mask]))}")

    report("KMeans", km)
    report("DBSCAN", db)
    if hlabels is not None:
        report("HDBSCAN", hlabels)

## End-to-end pipeline: UMAP → HDBSCAN

A common pattern: reduce noise and curse-of-dimensionality with UMAP, then cluster in the embedding. Requires `umap-learn` and `hdbscan`.

In [ ]:
if SKLEARN_OK and UMAP_OK and HDBSCAN_OK:
    from sklearn.metrics import adjusted_rand_score, silhouette_score

    Z = umap.UMAP(n_components=10, random_state=42).fit_transform(X)
    hz = hdbscan.HDBSCAN(min_cluster_size=40, min_samples=10).fit_predict(Z)
    mask = hz >= 0
    if mask.sum() > 1 and len(set(hz[mask])) > 1:
        sil_z = silhouette_score(Z[mask], hz[mask])
        ari_z = adjusted_rand_score(y_true[mask], hz[mask])
        print(f"UMAP(10d)+HDBSCAN ARI={ari_z:.3f} silhouette={sil_z:.3f}")
    else:
        print("Pipeline produced degenerate clustering; tune min_cluster_size / UMAP params.")
else:
    print("Skipping UMAP→HDBSCAN (install umap-learn and hdbscan).")

## Silhouette and ARI

- **Silhouette**: cohesion vs. separation in feature space (needs ≥2 clusters, ignores noise points if you filter `-1`).
- **ARI**: agreement with ground-truth labels when available (synthetic data here); invariant to permutation of cluster ids.

On real data, add **domain** checks—stability under bootstrap, cluster persistence in HDBSCAN—because no single score captures usability.

In [ ]:
if SKLEARN_OK:
    from sklearn.cluster import KMeans
    from sklearn.metrics import silhouette_score, adjusted_rand_score

    labels = KMeans(n_clusters=5, random_state=42, n_init="auto").fit_predict(X)
    sil = silhouette_score(X, labels)
    ari = adjusted_rand_score(y_true, labels)
    print({"silhouette": round(sil, 4), "ARI_vs_ground_truth": round(ari, 4)})
else:
    print("sklearn required for silhouette / ARI.")

## Conclusion

- Start with scaling and simple **KMeans** / **DBSCAN** baselines.
- Add **UMAP** when linear separation is poor but local structure matters.
- Prefer **HDBSCAN** when cluster density varies or you need a noise class (`-1`).
- Always validate with **multiple metrics** and qualitative inspection—especially before productionizing embeddings from deep models.